# Class 1: Text representation

Build a document-term matrix, compare documents, and see how changing the representation changes the result.

**In class:** work through Parts 1–3 in about 10 minutes. Part 4 is a short guided demonstration or an extension if you finish early.

The analysis uses only the Python standard library. Open this file in a notebook interface with a Python 3 kernel and run cells in order. The saved outputs show the original examples. After changing an input, rerun its cell and the cells that depend on it.

## 1. A matrix we can inspect

For the toy examples, lowercase each text and split at spaces. This keeps `ice-cream` as one token. Punctuation would stay attached to a word, so this deliberately simple rule is appropriate only for these controlled examples.

Before running the cell, count the tokens and the distinct types in the three sentences.

In [1]:
from collections import Counter
from math import sqrt

def toy_tokens(text):
    return text.lower().split()

def count_matrix(texts, vocabulary):
    counts = [Counter(toy_tokens(text)) for text in texts]
    return [[row[word] for word in vocabulary] for row in counts]

def show_table(headers, rows):
    """Print an aligned table without extra packages."""
    rows = [[str(value) for value in row] for row in rows]
    headers = [str(value) for value in headers]
    widths = [max(len(headers[i]), *(len(row[i]) for row in rows))
              for i in range(len(headers))]
    print("  ".join(value.ljust(width) for value, width in zip(headers, widths)))
    print("  ".join("-" * width for width in widths))
    for row in rows:
        print("  ".join(value.ljust(width) for value, width in zip(row, widths)))

texts = [
    "john loves ice-cream",
    "john loves oranges",
    "mary hates ice-cream",
]
vocabulary = ["john", "loves", "ice-cream", "oranges", "mary", "hates"]
X = count_matrix(texts, vocabulary)
show_table(["document"] + vocabulary,
           [[f"D{i + 1}"] + row for i, row in enumerate(X)])
print("\nRow totals:", [sum(row) for row in X])
print("Tokens:", sum(len(toy_tokens(text)) for text in texts))
print("Types:", len({token for text in texts for token in toy_tokens(text)}))

document  john  loves  ice-cream  oranges  mary  hates
--------  ----  -----  ---------  -------  ----  -----
D1        1     1      1          0        0     0    
D2        1     1      0          1        0     0    
D3        0     0      1          0        1     1    

Row totals: [3, 3, 3]
Tokens: 9
Types: 6


**Try a change.** In the cell above, replace the first sentence with `john loves loves ice-cream`.

1. Predict which cell changes and what the first row total becomes.
2. Rerun the cell and check your prediction.
3. Restore the original sentence when you finish.

The column order is a convention. What matters is that the same column refers to the same feature in every row. Our vocabulary covers every token in these examples. With a restricted vocabulary, row sums would count only the retained tokens.

## 2. Different sentences, identical counts

Read the following sentences. Who confronts whom in each sentence? Predict whether their count vectors will differ.

In [2]:
order_texts = ["Police confront protesters", "Protesters confront police"]
order_vocabulary = ["police", "confront", "protesters"]
order_X = count_matrix(order_texts, order_vocabulary)
show_table(["sentence"] + order_vocabulary,
           [[label] + row for label, row in zip(["A", "B"], order_X)])
print("\nSame vector:", order_X[0] == order_X[1])

sentence  police  confront  protesters
--------  ------  --------  ----------
A         1       1         1         
B         1       1         1         

Same vector: True


**Explain the result.** Which information is absent from this representation? Could a model that receives only these rows distinguish the two sentences?

Your answer:

## 3. Similarity depends on the representation

Use the two-word vocabulary from the slides: `banana`, `chocolate`.

We calculate cosine as the dot product divided by the product of the vector lengths. An all-zero vector has no direction, so this function reports an error for it rather than assigning a cosine.

In [3]:
def cosine(x, y):
    if len(x) != len(y):
        raise ValueError("Vectors must use the same feature space.")
    dot = sum(a * b for a, b in zip(x, y))
    norm_x = sqrt(sum(a * a for a in x))
    norm_y = sqrt(sum(b * b for b in y))
    if norm_x == 0 or norm_y == 0:
        raise ValueError("Cosine is undefined for an all-zero vector.")
    return dot / (norm_x * norm_y)

fruit_texts = [
    "chocolate chocolate chocolate banana",
    "banana banana banana banana chocolate",
    "banana banana",
]
fruit_vocabulary = ["banana", "chocolate"]
fruit_X = count_matrix(fruit_texts, fruit_vocabulary)
show_table(["text"] + fruit_vocabulary,
           [[f"T{i + 1}"] + row for i, row in enumerate(fruit_X)])

pairs = [(0, 1), (0, 2), (1, 2)]
print()
show_table(["pair", "count cosine"],
           [[f"T{i + 1}, T{j + 1}", f"{cosine(fruit_X[i], fruit_X[j]):.3f}"]
            for i, j in pairs])

text  banana  chocolate
----  ------  ---------
T1    1       3        
T2    4       1        
T3    2       0        

pair    count cosine
------  ------------
T1, T2  0.537       
T1, T3  0.316       
T2, T3  0.970       


### Repeating every word

Predict the cosine between Text 2 and a version that repeats all its words. Will Text 2's similarity with Text 1 change?

In [4]:
repeated_text = fruit_texts[1] + " " + fruit_texts[1]
repeated_vector = count_matrix([repeated_text], fruit_vocabulary)[0]
print("Original Text 2:", fruit_X[1])
print("Repeated Text 2:", repeated_vector)
print(f"Original vs repeated: {cosine(fruit_X[1], repeated_vector):.3f}")
print(f"Text 1 vs original Text 2: {cosine(fruit_X[0], fruit_X[1]):.3f}")
print(f"Text 1 vs repeated Text 2: {cosine(fruit_X[0], repeated_vector):.3f}")

Original Text 2: [4, 1]
Repeated Text 2: [8, 2]
Original vs repeated: 1.000
Text 1 vs original Text 2: 0.537
Text 1 vs repeated Text 2: 0.537


### Word counts or word presence?

A binary representation records `1` when a word occurs and `0` when it does not. It discards how often the word occurs.

Run the cell with `USE_BINARY = False`, then change it to `True` and rerun. Before the second run, predict the similarity of Texts 1 and 2.

In [5]:
USE_BINARY = False  # Change to True and rerun this cell.

chosen_X = ([[int(value > 0) for value in row] for row in fruit_X]
            if USE_BINARY else fruit_X)
print("Representation:", "binary presence" if USE_BINARY else "word counts")
show_table(["text"] + fruit_vocabulary,
           [[f"T{i + 1}"] + row for i, row in enumerate(chosen_X)])
print()
show_table(["pair", "cosine"],
           [[f"T{i + 1}, T{j + 1}", f"{cosine(chosen_X[i], chosen_X[j]):.3f}"]
            for i, j in pairs])

Representation: word counts
text  banana  chocolate
----  ------  ---------
T1    1       3        
T2    4       1        
T3    2       0        

pair    cosine
------  ------
T1, T2  0.537 
T1, T3  0.316 
T2, T3  0.970 


**Pause and explain.** Why do Texts 1 and 2 become identical in the binary representation? For which question would presence be useful? For which question would you want counts?

Your answer:

## 4. A small sample of real speeches

**Guided demo or extension.** The local CSV contains four complete speeches selected from the MethodsNET teaching corpus. Each row retains the source text, president, year, party, and speech type. The sample is for inspecting a measure. It cannot establish a historical trend or isolate a party effect.

Our question is: **How much attention do presidents give to employment?** Our first indicator counts the exact tokens `job` and `jobs` per 1,000 word tokens.

For these English texts, we lowercase the text and extract sequences of letters, retaining internal straight apostrophes and hyphens. Numbers and other punctuation are excluded. This is a different rule from the toy examples, and its effect on the denominator matters. No stop words are removed. We retain transcription material as it appears in the source.

In [6]:
from pathlib import Path
import csv
import re

# Works when the notebook starts in its folder or in the repository root.
relative_paths = [
    Path("../../data/sotu_sample.csv"),
    Path("data/sotu_sample.csv"),
]
data_path = next((path for path in relative_paths if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError("Open this notebook from its folder or the CEU_2026 root.")

with data_path.open(newline="", encoding="utf-8") as handle:
    speeches = list(csv.DictReader(handle))

def speech_tokens(text):
    return re.findall(r"[a-z]+(?:['-][a-z]+)*", text.lower())

TARGET_WORDS = {"job", "jobs"}
results = []
for speech in speeches:
    tokens = speech_tokens(speech["text"])
    mentions = sum(token in TARGET_WORDS for token in tokens)
    rate = 1000 * mentions / len(tokens)
    results.append([speech["year"], speech["president"], len(tokens),
                    mentions, f"{rate:.2f}"])
show_table(["year", "president", "tokens", "mentions", "per 1,000"], results)

year  president       tokens  mentions  per 1,000
----  --------------  ------  --------  ---------
2008  George W. Bush  5678    6         1.06     
2010  Barack Obama    7190    29        4.03     
2016  Barack Obama    6030    19        3.15     
2020  Donald Trump    6232    14        2.25     


### Reading the counted passages

Choose a year and read the sentences containing the target words. This simple sentence splitter is for inspection and can split incorrectly at abbreviations. A matched sentence can contain several mentions.

Would every matched passage count as employment attention under your definition? Could a passage discuss employment without either target word? Try adding `unemployment` to `TARGET_WORDS`, then rerun the table and the cell below. Does a higher count necessarily make the measure better?

In [7]:
YEAR_TO_INSPECT = "2010"
speech = next(row for row in speeches if row["year"] == YEAR_TO_INSPECT)
sentences = re.split(r"(?<=[.!?])\s+", speech["text"].strip())
matched = [sentence for sentence in sentences
           if TARGET_WORDS.intersection(speech_tokens(sentence))]
print(f"{speech['president']}, {speech['year']}: {len(matched)} matched passages\n")
for i, sentence in enumerate(matched, 1):
    print(f"{i}. {' '.join(sentence.split())}\n")

Barack Obama, 2010: 27 matched passages

1. The aspirations they hold are shared: a job that pays the bills, a chance to get ahead, most of all, the ability to give their children a better life.

2. Now, as we stabilized the financial system, we also took steps to get our economy growing again, save as many jobs as possible, and help Americans who had become unemployed.

3. And we're on track to add another 1 1/2 million jobs to this total by the end of the year.

4. The plan that has made all of this possible, from the tax cuts to the jobs, is the Recovery Act.

5. Economists on the left and the right say this bill has helped save jobs and avert disaster.

6. That is why jobs must be our number-one focus in 2010, and that's why I'm calling for a new jobs bill tonight.

7. Now, the true engine of job creation in this country will always be America's businesses.

8. We should start where most new jobs do, in small businesses, companies that begin when an entrepreneur takes a chance on a